linear probing with language's own direction

In [ ]:
from pathlib import Path
import torch

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
name = "qwen3-1.7b"
langs = ["eng_Latn", "npi_Deva", "srp_Cyrl", "srp_Latn"]
A = {lang: torch.load(ROOT / f"data/cache/e01/{name}/{lang}_acts.pt") for lang in langs}
print(A["npi_Deva"]["harmful_train"]["inst"].shape)

data prep

In [ ]:
def get_xy(lang, split, pos, layer):
    """X [N, d], y [N] (1 = harmful, 0 = harmless). split: "train" | "val"; pos: "inst" (h) | "last" (r)"""
    a = A[lang][f"harmful_{split}"][pos][:, layer]                 # [n1, L, d] -> [n1, d]
    b = A[lang][f"harmless_{split}"][pos][:, layer]                # [n0, d]
    X = torch.cat([a, b], dim=0)                                   # [n1 + n0, d]
    y = torch.cat([torch.ones(len(a)), torch.zeros(len(b))])      # [n1 + n0], float (BCE needs float targets)
    return X, y

def standardize(X_train, X_val):
    """z-score each dimension with TRAIN statistics only (val stats would leak)"""
    mu = X_train.mean(dim=0)                                       # [d]
    sd = X_train.std(dim=0) + 1e-6                                 # [d]
    return (X_train - mu) / sd, (X_val - mu) / sd                  # [N, d] - [d] broadcasts over rows

X_train, y_train = get_xy("eng_Latn", "train", "inst", 12)
X_val, y_val = get_xy("eng_Latn", "val", "inst", 12)
X_train, X_val = standardize(X_train, X_val)
print(X_train.shape, y_train.shape, X_val.shape, y_val.shape)     # [258, 2048] [258] [64, 2048] [64]

probe: torch logistic regression

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

def fit_probe(X_train, y_train, C=1.0, steps=1000, lr=1e-2, seed=0, verbose=False):
    """L2-regularized logistic regression, full batch. Same objective as sklearn LogisticRegression(C)
    divided by C*N: mean BCE + ||w||^2 / (2 C N). Bias not penalized."""
    torch.manual_seed(seed)
    probe = nn.Linear(X_train.shape[1], 1)                         # weight [1, d], bias [1]
    opt = torch.optim.Adam(probe.parameters(), lr=lr)
    lam = 1 / (2 * C * len(X_train))
    for step in range(steps):
        logits = probe(X_train).squeeze(-1)                        # [N, 1] -> [N]
        loss = F.binary_cross_entropy_with_logits(logits, y_train) + lam * (probe.weight ** 2).sum()
        opt.zero_grad()
        loss.backward()
        opt.step()
        if verbose and step % 100 == 0:
            print(step, round(loss.item(), 4))
    return probe

def accuracy(probe, X, y):
    with torch.no_grad():
        pred = (probe(X).squeeze(-1) > 0).float()                  # logit > 0  <=>  p > 0.5;  [N]
    return (pred == y).float().mean().item()

def auc(probe, X, y):
    """P(score of random harmful > score of random harmless); ties count 1/2. Threshold-free."""
    with torch.no_grad():
        s = probe(X).squeeze(-1)                                  # [N] logits
    pos, neg = s[y == 1], s[y == 0]                               # [n1], [n0] (boolean-mask indexing)
    gt = (pos[:, None] > neg[None, :]).float()                    # [n1, 1] vs [1, n0] -> [n1, n0]
    eq = (pos[:, None] == neg[None, :]).float()                   # ties
    return (gt + 0.5 * eq).mean().item()

probe = fit_probe(X_train, y_train, verbose=True)
print("train acc", accuracy(probe, X_train, y_train), "val acc", accuracy(probe, X_val, y_val))

checks: sklearn agreement, shuffled labels: shuffle the harmful and harmless prompts and train the probe which should give about half of the accuracy since there is randomness and we are just predicting randomness. 

In [ ]:
# check 1: same answer as sklearn on the same standardized arrays
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(C=1.0, max_iter=5000).fit(X_train.numpy(), y_train.numpy())
w = probe.weight.detach().squeeze(0)                               # [d]
print("sklearn val acc", clf.score(X_val.numpy(), y_val.numpy()),
      "| cos(torch w, sklearn w)", F.cosine_similarity(w, torch.tensor(clf.coef_[0]).float(), dim=0).item())

# check 2: shuffled-label control -> val acc should drop to ~0.5
g = torch.Generator().manual_seed(0)
probe_shuf = fit_probe(X_train, y_train[torch.randperm(len(y_train), generator=g)])
print("shuffled: train acc", accuracy(probe_shuf, X_train, y_train), "val acc", accuracy(probe_shuf, X_val, y_val))

multiple languages and probes

In [ ]:
from tqdm.auto import tqdm

n_layers = A[langs[0]]["harmful_train"]["inst"].shape[1]       # 28, read from data (not hardcoded)
positions = ["inst", "last"]                                    # h position (t_inst), r position (last post-inst)
src = "eng_Latn"                                                # transfer source language
assert langs[0] == src                                          # English probes must exist before transfer

acc = {}        # acc[(lang, pos)][kind] -> list of n_layers val accuracies; kind: own | shuffled | transfer
src_probe = {}  # src_probe[(pos, layer)] -> English probe, reused for transfer
for pos in positions:
    for lang in langs:
        r = {"own": [], "shuffled": [], "transfer": [], "own_auc": [], "transfer_auc": []}
        for layer in tqdm(range(n_layers), desc=f"{lang} {pos}", leave=False):
            X_tr, y_tr = get_xy(lang, "train", pos, layer)      # [258, d], [258]
            X_va, y_va = get_xy(lang, "val", pos, layer)        # [64, d], [64]
            X_tr, X_va = standardize(X_tr, X_va)                # both with THIS language's train stats

            probe = fit_probe(X_tr, y_tr)                       # own-language probe
            r["own"].append(accuracy(probe, X_va, y_va))
            r["own_auc"].append(auc(probe, X_va, y_va))
            if lang == src:
                src_probe[(pos, layer)] = probe

            g = torch.Generator().manual_seed(layer)            # control: same fit on scrambled labels
            y_shuf = y_tr[torch.randperm(len(y_tr), generator=g)]
            r["shuffled"].append(accuracy(fit_probe(X_tr, y_shuf), X_va, y_va))

            r["transfer"].append(accuracy(src_probe[(pos, layer)], X_va, y_va))   # English w on this language
            r["transfer_auc"].append(auc(src_probe[(pos, layer)], X_va, y_va))
        acc[(lang, pos)] = r
        print(f"{lang:9s} {pos:4s}  own L0 {r['own'][0]:.2f}  own max {max(r['own']):.2f} "
              f"(L{r['own'].index(max(r['own']))})  transfer max {max(r['transfer']):.2f}  shuffled mean "
              f"{sum(r['shuffled']) / n_layers:.2f}")

the plot

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, pos, title in zip(axes, positions, ["h position (t_inst)", "r position (last post-inst)"]):
    for i, lang in enumerate(langs):
        r = acc[(lang, pos)]
        ax.plot(r["own"], color=f"C{i}", label=f"{lang}")
        if lang != src:
            ax.plot(r["transfer"], color=f"C{i}", ls="--")
        ax.plot(r["shuffled"], color=f"C{i}", ls=":", alpha=0.4)
    ax.axhline(0.5, color="gray", lw=0.5)
    ax.set_title(f"probe val accuracy, {title}"); ax.set_xlabel("layer")
axes[0].set_ylabel("val accuracy (64 prompts)")
axes[1].legend(title="solid own | -- English->X | : shuffled", loc="lower right", fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, pos, title in zip(axes, positions, ["h position (t_inst)", "r position (last post-inst)"]):
    for i, lang in enumerate(langs[1:], start=1):                 # skip English (transfer = own)
        r = acc[(lang, pos)]
        ax.plot(r["transfer_auc"], color=f"C{i}", label=f"{lang} AUC")
        ax.plot(r["transfer"], color=f"C{i}", ls="--", alpha=0.6)
    ax.axhline(0.5, color="gray", lw=0.5)
    ax.set_title(f"English probe -> X, {title}"); ax.set_xlabel("layer")
axes[0].set_ylabel("val (64 prompts)")
axes[1].legend(title="solid AUC | -- accuracy", loc="lower right", fontsize=8)
plt.tight_layout(); plt.show()